# 🎯 TrackVision — Deploy on Google Colab

This notebook launches the **TrackVision** Streamlit app on Colab's GPU.

## 📁 Required File Structure in Google Drive

Before running, upload these files to your Google Drive under `MyDrive/Deep_Learning/`:

```
MyDrive/
└── Deep_Learning/
    ├── app.py                              ← Streamlit UI
    ├── tracker_engine.py                   ← Backend inference engine
    ├── requirements.txt                    ← Dependencies
    ├── fasterrcnn_mot16_finetuned.pth      ← Detector weights (~165 MB)
    └── siamese_reid_mot16.pth              ← ReID weights (~34 MB)
```

> **Note:** You do NOT need the MOT16 dataset folder or notebooks — just the 5 files above.

## 🚀 Steps
1. **Set runtime to GPU**: `Runtime → Change runtime type → T4 / A100`
2. **Run all cells below** (▶ Runtime → Run all)
3. **Click the public URL** printed at the bottom to open the app

---
## Step 1 — Verify GPU & Mount Google Drive

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change runtime type → select GPU")

from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Copy Project Files & Install Dependencies

In [ ]:
import shutil, os

# Source folder in Google Drive
DRIVE_DIR = "/content/drive/MyDrive/Deep_Learning"

# Working directory on the Colab VM (fast local disk)
WORK_DIR = "/content/trackvision"
os.makedirs(WORK_DIR, exist_ok=True)

# Files to copy
files = [
    "app.py",
    "tracker_engine.py",
    "requirements.txt",
    "fasterrcnn_mot16_finetuned.pth",
    "siamese_reid_mot16.pth",
]

for f in files:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join(WORK_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f"  ✅ {f} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ MISSING: {src}")

print("\nDone! All files copied to", WORK_DIR)

In [ ]:
# Install Streamlit and other deps (torch/torchvision already come with Colab)
!pip install -q streamlit pyngrok

---
## Step 3 — Quick Model Load Test

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

from tracker_engine import load_models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
detector, embed_model = load_models(device)
print(f"\n✅ Models loaded on {device}!")
print(f"   Detector params:  {sum(p.numel() for p in detector.parameters()):,}")
print(f"   ReID params:      {sum(p.numel() for p in embed_model.parameters()):,}")

# Cleanup to free memory before Streamlit starts
del detector, embed_model
torch.cuda.empty_cache()

---
## Step 4 — Launch Streamlit App

This will start the app and create a **public URL** you can open in any browser.

> ⏳ It may take 10-20 seconds for the URL to appear. Click it to open TrackVision!

In [ ]:
# ---- Option A: Using pyngrok (reliable, needs free ngrok account) ----
# Sign up at https://ngrok.com and get your auth token (free tier is fine)
# Then uncomment the two lines below and paste your token:

# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN_HERE")  # paste your token

# ---- Option B: Using localtunnel (no signup needed) ----
# This is the default — works without any account

import subprocess, time, threading, re

os.chdir(WORK_DIR)

# Start Streamlit in background
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.address", "0.0.0.0",
     "--server.fileWatcherType", "none"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

# Wait for Streamlit to be ready
time.sleep(8)
print("✅ Streamlit server started on port 8501")

# --- Try pyngrok first, fallback to localtunnel ---
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(8501)
    print(f"\n🌐 Your app is live at: {public_url}")
except Exception:
    print("ngrok not configured — using localtunnel instead...")
    !npm install -g localtunnel > /dev/null 2>&1
    print("\n🌐 Starting tunnel... Click the URL below when it appears:\n")
    !npx localtunnel --port 8501

---
## 📖 Troubleshooting

| Issue | Fix |
|---|---|
| `❌ MISSING` file | Upload it to `MyDrive/Deep_Learning/` and re-run Step 2 |
| `No GPU detected` | `Runtime → Change runtime type → T4 GPU` → restart |
| localtunnel says 'password' | Enter the IP shown on the tunnel page (it's a spam filter) |
| ngrok error | Sign up at ngrok.com, get token, paste it in Step 4 |
| App is slow | Make sure you selected **A100** GPU in runtime settings |